# 규칙기반 이름 변환 코드 (개인 프로젝트로)

In [ ]:
# language: python
# Purpose: Demonstration implementation for converting names (English, Chinese, Japanese)
# to consonant-based Hangul initials, with editable mapping tables and a small Korean name DB example.
# This code will try to use phonemizer, pypinyin, and jaconv if available; otherwise fall back to
# simple heuristic rules so it still runs without those libraries.

from typing import List, Tuple, Dict
import re

# --- Try imports; if not available, note and continue with fallbacks ---
_HAS_PHONEMIZER = False
_HAS_PYPINYIN = False
_HAS_JACONV = False

try:
    from phonemizer import phonemize
    _HAS_PHONEMIZER = True
except Exception as e:
    phonemize = None
    print("phonemizer not available; falling back to heuristic for English.")

try:
    from pypinyin import lazy_pinyin, Style
    _HAS_PYPINYIN = True
except Exception as e:
    lazy_pinyin = None
    Style = None
    print("pypinyin not available; falling back to heuristic for Chinese.")

try:
    import jaconv
    _HAS_JACONV = True
except Exception as e:
    jaconv = None
    print("jaconv not available; falling back to heuristic for Japanese.")


# -------------------------
# Editable mapping tables
# -------------------------

# 우선 순위로 긴 토큰을 위주로 둠 (digraphs/trigraphs 먼저 처리)
CONSONANT_TO_CHOSEONG = {
    # common multi-letter clusters (digraphs/trigraphs)
    'sch': 'ㅅ',   # e.g. 'Sch' (독일계) -> ㅅ
    'shr': 'ㅅ',   # rare
    'thr': 'ㅌ',   # treat as 'th'+'r' -> ㅌ or ㅅ 선택 가능
    'ts':  'ㅊ',   # e.g. 'Tsvet' -> ㅊ (선택적)
    'tz':  'ㅈ',   # e.g. 'Tz' translit
    'ch':  'ㅊ',   # 'ch' -> ㅊ (Jonathan: j->ㅈ, ch->ㅊ)
    'sh':  'ㅅ',   # 'sh' -> ㅅ (혹은 ʃ -> ㅅ)
    'zh':  'ㅈ',   # romanization of ʒ/zh (pinyin zh -> ㅈ)
    'ng':  'ㅇ',   # initial 'ng' rare; middle 'ng' 종성으로 고려 가능
    'gh':  'ㄱ',   # 'gh' often /g/ or /f/ (ex. 'enough' 예외)
    'ph':  'ㅍ',   # greek-origin -> /f/ 소리지만 ㅍ 권장 (또는 ㅂ 매핑 옵션)
    'gn':  'ㄴ',   # 'gn' -> n (g often silent in 'gn' initial)
    'kn':  'ㄴ',   # 'kn' (k silent) -> n
    'wr':  'ㄹ',   # 'wr' -> r/wr -> ㄹ
    'ps':  'ㅅ',   # e.g. 'Psh' -> ㅅ  (p often silent in ps-)
    'qu':  'ㅋ',   # 'qu' -> /kw/ ~ ㅋ (or ㅋ+ㅜ 처리)
    'quw': 'ㅋ',   # rare tri combos
    'ks':  'ㄱ',   # 'x' -> ks -> map to ㄱ or ㄱㅅ (single초성이면 ㄱ)
    'sc':  'ㅅ',   # 'sc' in 'sc' before e/i -> ㅅ (science -> ㅅ)
    # single-letter mappings (fallback)
    'b': 'ㅂ', 'c': 'ㅋ', 'd': 'ㄷ', 'f': 'ㅍ', 'g': 'ㄱ',
    'h': 'ㅎ', 'j': 'ㅈ', 'k': 'ㅋ', 'l': 'ㄹ', 'm': 'ㅁ',
    'n': 'ㄴ', 'p': 'ㅂ', 'q': 'ㅋ', 'r': 'ㄹ', 's': 'ㅅ',
    't': 'ㅌ', 'v': 'ㅂ', 'w': 'ㅇ', 'x': 'ㄱ', 'y': 'ㅇ', 'z': 'ㅈ'
}

# You can adjust priority or add language-specific overrides if desired.
# -------------------------
# Small example Korean name "DB"
# Each entry: (full_name, initial_consonant_sequence)
# initial_consonant_sequence is the 초성 of the name, e.g. "홍록기" -> "ㅎㄹㄱ"
# In a real DB, you could precompute 초성 and index by it for fast lookup.
KOREAN_NAME_DB = [
    ("홍록기", "ㅎㄹㄱ"),
    ("김덕수", "ㄱㄷㅅ"),
    ("박도현", "ㅂㄷㅎ"),
    ("이태윤", "ㅇㅌㅇ"),
    ("남궁성", "ㄴㄱㅅ"),

    ("최영훈", "ㅊㅇㅎ"),
    ("정민수", "ㅈㅁㅅ"),
    ("손정우", "ㅅㅈㅇ"),
    ("오세훈", "ㅇㅅㅎ"),
    ("백현수", "ㅂㅎㅅ"),
]

# Helper to compute 초성 for a given Hangul name string (simple extraction)
# This uses Unicode decomposition to find the initial consonant index.
CHOSEONG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def hangul_initials(name: str) -> str:
    """Return the sequence of Korean initial consonants (초성) for a Hangul name.
    Non-Hangul characters are skipped.
    Example: "홍록기" -> "ㅎㄹㄱ"
    """
    result = []
    for ch in name:
        code = ord(ch)
        if 0xAC00 <= code <= 0xD7A3:
            s_index = code - 0xAC00
            chosung_index = s_index // (21 * 28)
            result.append(CHOSEONG_LIST[chosung_index])
        else:
            # non-hangul character -> skip or keep placeholder
            pass
    return "".join(result)


# -------------------------
# Language-specific helpers
# -------------------------

def english_to_phonemes_fallback(name: str) -> str:
    """Very simple fallback: normalize letters and return a 'pseudo-phoneme' string.
    This is NOT accurate IPA; it's a heuristic roman-letter based phoneme approximation,
    intended to allow consonant extraction even without phonemizer.
    """
    n = name.lower()
    # keep letters and spaces only
    n = re.sub(r"[^a-z\s'-]", "", n)
    # # common clusters to preserve
    # n = re.sub(r"ch", "ch", n)
    # n = re.sub(r"sh", "sh", n)
    # collapse repeated
    n = re.sub(r"\s+", " ", n).strip()
    return n

def extract_consonant_sequence_from_roman(roman: str) -> List[str]:
    """Extract consonant 'tokens' from a romanized string.
    This is a heuristic: we look for common consonant clusters first (ch, sh, th, ng, ph, kn, qu)
    and then single letters. Vowels are ignored.
    """
    tokens = []
    roman = roman.lower()
    # preserve common two-letter clusters
    clusters = ['ch','sh','th','ng','ph','kn','qu','ts']
    tri_clusters = ['sch', 'shr', 'thr']
    i = 0
    while i < len(roman):
        if roman[i].isspace() or roman[i] in ["'", "-"]:
            i += 1
            continue
        matched = False
        # try tri-clusters
        if i + 2 < len(roman):
            tri_pair = roman[i:i+3]
            if tri_pair in tri_clusters:
                tokens.append(tri_pair)
                i += 3
                continue
        # clusters
        if i + 1 < len(roman):
            pair = roman[i:i+2]
            if pair in clusters:
                tokens.append(pair)
                i += 2
                continue
        # single consonant
        ch = roman[i]
        if ch in 'bcdfghjklmnpqrstvwxyz':
            tokens.append(ch)
        # else vowel -> skip
        i += 1
    return tokens

def roman_tokens_to_choseong(tokens: List[str], mapping: Dict[str,str]) -> List[str]:
    """Map roman consonant tokens to Hangul initial consonants using mapping table."""
    result = []
    for t in tokens:
        if t in mapping:
            result.append(mapping[t])
        else:
            # try to map by first character
            if t and t[0] in mapping:
                result.append(mapping[t[0]])
            else:
                # if unknown, append placeholder 'ㅇ' (could be changed)
                result.append('ㅇ')
    return result

# English pipeline
def name_to_initials_en(name: str, user_mapping: Dict[str,str]=None) -> str:
    """Convert English name to 초성 sequence (string of Hangul initials)."""
    if _HAS_PHONEMIZER:
        try:
            # attempt to produce IPA-like transcription (may produce different formats depending on phonemizer settings)
            phon = phonemize(name, language='en-us', backend='espeak', strip=True, with_stress=False)
            # phonemize may produce multiple tokens; keep letters and basic IPA symbols
            roman = phon
        except Exception:
            roman = english_to_phonemes_fallback(name)
    else:
        roman = english_to_phonemes_fallback(name)

    tokens = extract_consonant_sequence_from_roman(roman)
    mapping = CONSONANT_TO_CHOSEONG.copy()
    if user_mapping:
        mapping.update(user_mapping)
    chos = roman_tokens_to_choseong(tokens, mapping)
    # compress to two consonants: choose first and last non-ㅇ by default
    chos_non_null = [c for c in chos if c != 'ㅇ']
    if len(chos_non_null) >= 2:
        picked = chos_non_null[0:2]
    elif len(chos_non_null) == 1:
        picked = [chos_non_null[0]]
    else:
        # fallback: take first two from chos (even if ㅇ)
        picked = chos[:2]
    return "".join(picked)

# Chinese pipeline
def name_to_initials_zh(name: str, user_mapping: Dict[str,str]=None) -> str:
    """Convert Chinese name to 초성 sequence via pinyin initials."""
    mapping = CONSONANT_TO_CHOSEONG.copy()
    if user_mapping:
        mapping.update(user_mapping)

    if _HAS_PYPINYIN:
        try:
            pinyins = lazy_pinyin(name, style=Style.NORMAL)  # e.g. ['zhang','wei']
        except Exception:
            # fallback to naive character->pinyin heuristic
            pinyins = []
            for ch in name:
                # ASCII letters pass-through
                if ord(ch) < 128:
                    pinyins.append(ch)
                else:
                    pinyins.append('?')
    else:
        # naive fallback: replace CJK chars with placeholder letters using basic romanization heuristics,
        # For illustration we treat each CJK char as 'zh' initial or '?'
        pinyins = []
        for ch in name:
            if re.match(r'[\u4e00-\u9fff]', ch):
                # try to approximate by the Unicode block - we can't do real pinyin without pypinyin
                pinyins.append('zh')  # placeholder
            else:
                pinyins.append(ch)

    # Extract initials from pinyin tokens (leading consonant letters)
    initials = []
    for p in pinyins:
        # take leading consonant cluster (letters before first vowel)
        m = re.match(r'^([^aeiouy]+)', p.lower())
        if m:
            initials.append(m.group(1))
        else:
            # vowel-initial syllable => treat as 'ㅇ' (ng/zero-initial)
            initials.append('')
    # Map initials to choseong
    chos_list = []
    for ini in initials:
        if ini == '':
            chos_list.append('ㅇ')
        else:
            # map common pinyin initials
            # handle multi-letter initials like 'zh','ch','sh'
            if ini in mapping:
                chos_list.append(mapping[ini])
            else:
                # try first letter
                if ini[0] in mapping:
                    chos_list.append(mapping[ini[0]])
                else:
                    chos_list.append('ㅇ')
    # compress to two consonants
    chos_non_null = [c for c in chos_list if c != 'ㅇ']
    if len(chos_non_null) >= 2:
        picked = chos_non_null[0:2]
    elif len(chos_non_null) == 1:
        picked = [chos_non_null[0]]
    else:
        picked = chos_list[:2]
    return "".join(picked)

# Japanese pipeline
def name_to_initials_ja(name: str, user_mapping: Dict[str,str]=None) -> str:
    """Convert Japanese name (kana or romaji) to 초성 sequence."""
    mapping = CONSONANT_TO_CHOSEONG.copy()
    if user_mapping:
        mapping.update(user_mapping)

    # If jaconv available, attempt to romanize kana into ascii romaji
    if _HAS_JACONV:
        try:
            # jaconv may expect kana; try to convert kana to romaji
            # Note: jaconv has function kana2alphabet or kata2alphabet depending on version; use fallback via hira->romanization
            # We'll try a few common functions safely.
            if hasattr(jaconv, "kana2alphabet"):
                rom = jaconv.kana2alphabet(name)
            elif hasattr(jaconv, "kana2alphabet"):  # redundant but safe
                rom = jaconv.kana2alphabet(name)
            elif hasattr(jaconv, "kata2alphabet"):
                rom = jaconv.kata2alphabet(name)
            else:
                # As last resort, try to convert katakana to hiragana then to romaji using simple replace
                rom = name
        except Exception:
            rom = name
    else:
        # fallback: if contains kana (hiragana/katakana), try simple transliteration of common kana patterns
        # Very small heuristic: replace common kana with romaji
        kana_to_romaji_examples = {
            'た': 'ta', 'ろ': 'ro', 'う': 'u',
            'たろう': 'tarou', 'たろ': 'taro',
            'さ': 'sa', 'と': 'to', 'し': 'shi', 'や': 'ya'
        }
        rom = name
        for k,v in kana_to_romaji_examples.items():
            rom = rom.replace(k, v)
        # if still contains non-ascii, strip them (best-effort)
        rom = re.sub(r'[^\x00-\x7f]', '', rom)

    # Use roman pipeline to extract consonant tokens
    tokens = extract_consonant_sequence_from_roman(rom)
    chos = roman_tokens_to_choseong(tokens, mapping)
    chos_non_null = [c for c in chos if c != 'ㅇ']
    if len(chos_non_null) >= 2:
        picked = chos_non_null[0:2]
    elif len(chos_non_null) == 1:
        picked = [chos_non_null[0]]
    else:
        picked = chos[:2]
    return "".join(picked)


# -------------------------
# Matcher: find a Korean name from DB by 초성
# -------------------------
def find_korean_name_by_initials(initials: str, db: List[Tuple[str,str]]) -> str:
    """Given initials like 'ㄹㄱ', find the best matching Korean name from db.
    Matching strategy (simple demo):
      1. exact prefix match (name 초성 starts with initials)
      2. contains initials anywhere
      3. startswith first initial and second consonant anywhere
      4. fallback: first name in DB
    """
    initials = initials or ''
    # 1. exact prefix
    for name, chos in db:
        if chos.startswith(initials):
            return name
    # 2. contains
    for name, chos in db:
        if initials in chos:
            return name
    # 3. first then anywhere
    if len(initials) >= 1:
        f = initials[0]
        for name, chos in db:
            if chos.startswith(f):
                return name
    # fallback
    return db[0][0] if db else ""

# -------------------------
# High-level API
# -------------------------
def map_name_to_korean(name: str, lang_hint: str=None, user_mapping: Dict[str,str]=None, db=None) -> Tuple[str,str]:
    """Top-level function.
    - name: input name string
    - lang_hint: optional 'en', 'zh', or 'ja'. If None, we try to guess.
    - user_mapping: optional mapping overrides for consonant->초성 mapping
    - db: optional Korean name DB list; if None, use KOREAN_NAME_DB
    Returns (initials, matched_korean_name)
    """
    if db is None:
        db = KOREAN_NAME_DB

    # naive language guess if not provided
    if lang_hint is None:
        # if contains CJK Unified Ideographs -> zh or ja (we'll assume zh for han characters)
        if re.search(r'[\u4e00-\u9fff]', name):
            lang = 'zh'
        # hiragana/katakana -> ja
        elif re.search(r'[\u3040-\u30ff]', name):
            lang = 'ja'
        else:
            # default english/latin input
            lang = 'en'
    else:
        lang = lang_hint

    if lang == 'en':
        initials = name_to_initials_en(name, user_mapping)
    elif lang == 'zh':
        initials = name_to_initials_zh(name, user_mapping)
    elif lang == 'ja':
        initials = name_to_initials_ja(name, user_mapping)
    else:
        initials = name_to_initials_en(name, user_mapping)

    matched = find_korean_name_by_initials(initials, db)
    return initials, matched

# -------------------------
# Demo examples
# -------------------------
examples = [
    ("Lady Gaga", None),
    ("Dustin", None),
    ("张伟", None),
    ("たろう", None),
    ("Tarou", "ja"),
    ("Al", None),
    ("Chloe", None),
    ("김민수", None),  # already Korean - should be detected as Han or Hangul
]

print("=== Demo mapping results ===")
for name, hint in examples:
    initials, matched = map_name_to_korean(name, lang_hint=hint)
    print(f"Input: {name:10s} | LangHint: {hint or 'auto':4s} | Initials: {initials:4s} | Matched Korean name: {matched}")

# Provide instructions and examples for editing mapping and DB
print("\n--- How to edit mapping ---")
print("CONSONANT_TO_CHOSEONG is a dict you can modify. Example entries shown:")
for k,v in list(CONSONANT_TO_CHOSEONG.items())[:12]:
    print(f"  '{k}': '{v}'")

print("\n--- How to extend Korean name DB ---")
print("Each entry should be a tuple: (full_name_hangul, precomputed_initials).")
print("You can compute initials with hangul_initials(name). Example:")
for name,chos in KOREAN_NAME_DB[:6]:
    print(f"  {name} -> {chos}")

# show how to add a new Korean name and recompute
new_name = "김록기"
print("\nExample: adding a new name to DB and recomputing initials:")
print("  new_name =", new_name, "-> initials:", hangul_initials(new_name))

# Provide a small helper to generate DB entries from a list of Hangul names
def build_db_from_names(hangul_names: List[str]) -> List[Tuple[str,str]]:
    return [(n, hangul_initials(n)) for n in hangul_names]

print("\nYou can build DB quickly with build_db_from_names(['홍록기','김덕수','박도현']) ->", build_db_from_names(['홍록기','김덕수','박도현']))

# Expose functions for user to reuse in notebook environment
__all__ = [
    "map_name_to_korean",
    "name_to_initials_en",
    "name_to_initials_zh",
    "name_to_initials_ja",
    "CONSONANT_TO_CHOSEONG",
    "KOREAN_NAME_DB",
    "hangul_initials",
    "build_db_from_names",
]

